# Vegetation Data Access

Accessing NDVI data

# STEP 2: AppEEARS API

# STEP 0: Set up

To get started on this notebook, you’ll need to restore any variables
from previous notebooks to your workspace. To save time and memory, make
sure to specify which variables you want to load.

In [1]:
%store -r

In [1]:
%pip install pystac-client planetary-computer --quiet

Note: you may need to restart the kernel to use updated packages.


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
# Import libraries
import pystac_client
import planetary_computer
import rioxarray as rxr
import xarray as xr
import pandas as pd
import hvplot.xarray

## Exploring the AppEEARS API for NASA Earthdata access

Before you get started with the data download today, you will need a
free [NASA Earthdata account](https://urs.earthdata.nasa.gov/home) if
you don’t have one already!

Over the next four cells, you will download MODIS NDVI data for the
study period. MODIS is a multispectral instrument that measures Red and
NIR data (and so can be used for NDVI). There are two MODIS sensors on
two different platforms: satellites Terra and Aqua.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-read"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Read More</div></div><div class="callout-body-container callout-body"><p><a href="https://modis.gsfc.nasa.gov/">Learn more about MODIS
datasets and the science they support</a></p></div></div>

Since we’re asking for a special download that only covers our study
area, we can’t just find a link to the data - we have to negotiate with
the data server. We’re doing this using the
[APPEEARS](https://appeears.earthdatacloud.nasa.gov/api/) API
(Application Programming Interface). The API makes it possible for you
to request data using code. You can use code from the `earthpy` library
to handle the API request.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Often when we want to do something more complex in coding we find an
example and modify it. This download code is already almost a working
example. Your task will be:</p>
<ol type="1">
<li>Replace the start and end dates in the task parameters. Download
data from July, when greenery is at its peak in the Northern
Hemisphere.</li>
<li>Replace the year range. You should get 3 years before and after the
event so you can see the change!</li>
<li>Replace <code>gdf</code> with the name of <strong>your</strong> site
geodataframe.</li>
<li><strong>Enter your NASA Earthdata username and password when
prompted.</strong> The prompts can be a little hard to see – look at the
top of your screen!</li>
</ol></div></div>

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-respond"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Reflect and Respond</div></div><div class="callout-body-container callout-body"><p>What would the product and layer name be if you were trying to
download Landsat Surface Temperature Analysis Ready Data (ARD) instead
of MODIS NDVI?</p></div></div>

> **Important**
>
> It can take some time for Appeears to process your request - anything
> from a few minutes to a few hours depending on how busy they are. You
> can check your progress by:
>
> 1.  Going to the [Appeears
>     webpage](https://appeears.earthdatacloud.nasa.gov/)
> 2.  Clicking the `Explore` tab
> 3.  Logging in with your Earthdata account

In [3]:
#Connect to the Catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [5]:
%store -r boundary_gdf
boundary_gdf

,OBJECTID,NBHD_ID,NBHD_NAME,TYPOLOGY,NOTES,GLOBALID,geometry
32,35,49,Northeast Park Hill,,Removed NEST Status May 2023,c05fc42c-3b3e-4322-a31b-7cc7371c8cda,"POLYGON ((-104.90351 39.78382, -104.90353 39.7..."


In [7]:
#Search for matching images in both years
search_2000 = catalog.search(
    collections=["modis-13Q1-061"],
    intersects=boundary_gdf.geometry.iloc[0],
    datetime=f"{start_year}-06-01/{start_year}-09-01"
)
items_2000 = list(search_2000.items())

search_2025 = catalog.search(
    collections=["modis-13Q1-061"],
    intersects=boundary_gdf.geometry.iloc[0],
    datetime=f"{end_year}-06-01/{end_year}-09-01"
)
items_2025 = list(search_2025.items())

items = items_2000 + items_2025
len(items_2000), len(items_2025), len(items)

(7, 6, 13)

In [8]:
items[0].assets.keys()

dict_keys(['hdf', 'metadata', '250m_16_days_EVI', '250m_16_days_NDVI', '250m_16_days_VI_Quality', '250m_16_days_MIR_reflectance', '250m_16_days_NIR_reflectance', '250m_16_days_red_reflectance', '250m_16_days_blue_reflectance', '250m_16_days_sun_zenith_angle', '250m_16_days_pixel_reliability', '250m_16_days_view_zenith_angle', '250m_16_days_relative_azimuth_angle', '250m_16_days_composite_day_of_the_year', 'tilejson', 'rendered_preview'])

In [10]:
items[0].properties

{'created': '2020-02-20T06:35:31Z',
 'updated': '2020-04-02T09:34:09.316000Z',
 'datetime': None,
 'platform': 'terra',
 'proj:wkt2': 'PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]',
 'proj:shape': [4800, 4800],
 'instruments': ['modis'],
 'end_datetime': '2000-09-12T23:59:59Z',
 'modis:tile-id': '51009005',
 'proj:geometry': {'type': 'Polygon',
  'coordinates': [[[-8895604.157333, 3335851.559],
    [-8895604.157333, 4447802.078667],
    [-10007554.677, 4447802.078667],
    [-10007554.677, 3335851.559],
    [-8895604.157333, 3335851.559]]]},
 'proj:transform': [231.65635826395825,
  0.0,
  -10007554.677,
  0.0,
  

In [11]:
date = pd.to_datetime(item.properties['start_datetime']).tz_localize(None)

In [12]:
ndvi_das = []

for item in items:
    # Open the NDVI band directly from the cloud, no download needed
    da = rxr.open_rasterio(item.assets['250m_16_days_NDVI'].href, masked=True).squeeze()

    # Clip to Park Hill boundary (reproject boundary to match the raster's CRS first)
    boundary_reprojected = boundary_gdf.to_crs(da.rio.crs)
    da = da.rio.clip(boundary_reprojected.geometry)

    # MODIS NDVI is scaled by 10000; apply the correction
    da = da * 0.0001

    # Add date as a dimension
    date = pd.to_datetime(item.properties['start_datetime']).tz_localize(None)
    da = da.assign_coords({'date': date})
    da = da.expand_dims({'date': 1})
    da.name = 'NDVI'

    ndvi_das.append(da)

len(ndvi_das)

13

In [13]:
ndvi_da = xr.combine_by_coords(ndvi_das, coords=['date'])
ndvi_da

/tmp/ipykernel_2282/3793966472.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da = xr.combine_by_coords(ndvi_das, coords=['date'])
/tmp/ipykernel_2282/3793966472.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da = xr.combine_by_coords(ndvi_das, coords=['date'])


<xarray.Dataset> Size: 18kB
Dimensions:      (date: 13, y: 14, x: 24)
Coordinates:
    band         int64 8B 1
  * x            (x) float64 192B -8.97e+06 -8.969e+06 ... -8.965e+06 -8.964e+06
  * y            (y) float64 112B 4.425e+06 4.424e+06 ... 4.422e+06 4.422e+06
    spatial_ref  int64 8B 0
  * date         (date) datetime64[ns] 104B 2000-05-24 2000-06-09 ... 2025-06-26
Data variables:
    NDVI         (date, y, x) float32 17kB nan nan nan nan ... nan nan nan nan

In [14]:
%store ndvi_da boundary_gdf start_year end_year

Stored 'ndvi_da' (Dataset)
Stored 'boundary_gdf' (GeoDataFrame)
Stored 'start_year' (int)
Stored 'end_year' (int)


start_year = 2000
end_year = 2025
download_key = 'park-hill-ndvi'

download_key = 'park-hill-ndvi-2000'

ndvi_downloader_2000 = eaapp.AppeearsDownloader(
    download_key=download_key,
    product='MOD13Q1.061',
    layer='_250m_16_days_NDVI',
    start_date='06-01',
    end_date='09-01',
    recurring=True,
    year_range=[2000, 2000],
    polygon=boundary_gdf
)
ndvi_downloader_2000.download_files(cache=True)

ndvi_downloader_2000.download_files(cache=False)

## Putting it together: Working with multi-file raster datasets in Python

Now you need to load all the downloaded files into Python. You may have
noticed that the \`earthpy.appears module gives us all the downloaded
file names…but only some of those are the NDVI files we want while
others are quality files that tell us about the confidence in the
dataset. For now, the files we want all have “NDVI” in the name.

Let’s start by getting all the NDVI file names. You will also need to
extract the date from the filename. Check out [the lesson on getting
information from filenames in the
textbook](https://www.earthdatascience.org/courses/intro-to-earth-data-science/write-efficient-python-code/loops/data-workflows-with-loops/).
We’re using a slightly different method here (the `.rglob()` or
**recursive** glob method, which searchs all the directories nested
inside the path), but the principle is the same.

> **GOTCHA ALERT!**
>
> `glob` doesn’t necessarily find files in the order you would expect.
> Make sure to **sort** your file names like it says in the textbook.

### Repeating tasks in Python

Now you should have a few dozen files! For each file, you need to:

-   Load the file in using the `rioxarray` library
-   Get the date from the file name
-   Add the date as a dimension coordinate
-   Give your data variable a name

You don’t want to write out the code for each file! That’s a recipe for
**copy pasta** and errors. Luckily, Python has tools for doing similar
tasks repeatedly. In this case, you’ll use one called a `for` loop.

There’s some code below that uses a `for` loop in what is called an
**accumulation pattern** to process each file. That means that you will
save the results of your processing to a list each time you process the
files, and then merge all the arrays in the list.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ul>
<li>Look at the file names. How many characters from the end is the
date? <code>doy_start</code> and <code>doy_end</code> are used to
extract the day of the year (doy) from the file name. You will need to
count characters from the end and change the values to get the right
part of the file name. HINT: the index -1 in Python means the last
value, -2 second-to-last, and so on.</li>
<li>Replace any required variable names with your chosen variable
names</li>
</ul></div></div>

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Next, stack your arrays by date into a time series:</p>
<ol type="1">
<li>Modify the code to match your prior workflow steps and to use
descriptive variable names</li>
<li>Replace <code>coordinate_name</code> with the actual name of the
coordinate you want to build up.</li>
</ol></div></div>

# STEP -1: Wrap up

Don’t forget to store your variables so you can use them in other
notebooks! Replace `var1` and `var2` with the variable you want to save,
separated by spaces.

In [12]:
%store var1 var2

Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!